In [ ]:
#| include: false
import os, logging, warnings, contextlib, io
os.environ['TQDM_DISABLE'] = '1'
warnings.filterwarnings('ignore')
warnings.filterwarnings('error', message='.*looks like a percent.*')
logging.getLogger('torch.onnx._internal.exporter._schemas').setLevel(logging.ERROR)
with contextlib.redirect_stderr(io.StringIO()):
    try: import torchao  # noqa: F401
    except Exception: pass

## Overview

One model through the four steps fasterai composes: `SparsifyCallback` zeroes weights during training,
`Pruner` removes filters, `Quantizer` lowers the arithmetic to INT8, `export_qdq` writes an ONNX file a
runtime can read. Every measured number below is printed by a cell on this page.

In [ ]:
from fastai.vision.all import *
from fasterai.sparse.all import *
from fasterai.prune.all import *
from fasterai.core.all import quant_spec
from fasterai.quantize.quantizer import Quantizer
from fasterai.export.all import export_onnx, export_qdq, qdq_stats, verify_qdq, ONNXModel

import tempfile, torch_pruning as tp
from math import sqrt

tmp = Path(tempfile.mkdtemp())   # every file this page writes goes here

In [ ]:
import onnx, onnxruntime
print(f"torch {torch.__version__} | onnx {onnx.__version__} | onnxruntime {onnxruntime.__version__} "
      f"| training on {default_device()}")

torch 2.9.1+cu128 | onnx 1.17.0 | onnxruntime 1.24.1 | training on cuda:0


## 1. Data and baseline

The cat/dog labels of the PETS dataset, images resized to 64 px.

In [ ]:
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")

def label_func(f): return f[0].isupper()

dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(64))

labels = [label_func(f.name) for f in dls.valid.items]
print(f"{len(dls.train_ds)} training images, {len(dls.valid_ds)} validation images, majority class "
      f"{max(sum(labels), len(labels)-sum(labels))/len(labels):.2%}")

5912 training images, 1478 validation images, majority class 66.91%


In [ ]:
def report(learn, name):
    "Validation accuracy with its Wilson 95% interval"
    n = len(learn.dls.valid_ds)
    with learn.no_bar(): acc = float(learn.validate()[1])
    k, z = round(acc*n), 1.96
    p, d = k/n, 1 + z**2/n
    c = p + z**2/(2*n)
    h = z*sqrt(p*(1-p)/n + z**2/(4*n**2))
    print(f"{name}: {acc:.2%} ({k}/{n}), Wilson 95% [{(c-h)/d:.2%}, {(c+h)/d:.2%}]")
    return acc

stages = []

def record(name, model, acc=None):
    "Parameters, MACs at 64 px, state_dict size on disk and share of zeros in the convolutions"
    model.eval()
    macs, params = tp.utils.count_ops_and_params(
        model, torch.randn(1, 3, 64, 64).to(next(model.parameters()).device))
    f = tmp/f"{name}.pth"; torch.save(model.state_dict(), f)
    w = [m.weight for m in model.modules() if isinstance(m, nn.Conv2d)]
    zeros = sum(int((x == 0).sum()) for x in w) / sum(x.numel() for x in w)
    stages.append(dict(stage=name, params=params, macs=macs, mb=f.stat().st_size/1e6,
                       zeros=zeros, acc=acc))
    print(f"{name}: {params/1e6:.2f} M parameters, {macs/1e6:.0f} MMACs, "
          f"{f.stat().st_size/1e6:.1f} MB state_dict, {zeros:.1%} zeros in conv weights")

A ResNet-18 with its ImageNet weights, fine-tuned for three epochs:

In [ ]:
learn = vision_learner(dls, resnet18, metrics=accuracy, concat_pool=False)
learn.unfreeze()
learn.fit_one_cycle(3)

epoch,train_loss,valid_loss,accuracy,time
0,0.647689,0.335134,0.858593,00:04
1,0.336341,0.254721,0.893099,00:04
2,0.187006,0.204517,0.922192,00:04


In [ ]:
record("baseline", learn.model, report(learn, "baseline"))

baseline: 92.22% (1363/1478), Wilson 95% [90.74%, 93.48%]
baseline: 11.44 M parameters, 149 MMACs, 45.9 MB state_dict, 0.0% zeros in conv weights


`concat_pool=False` gives the head a plain average pool: the exporter of step 5 has no translation
for adaptive max pooling.

## 2. Sparsify

`SparsifyCallback` zeroes weights while training continues:

- **`sparsity=0.5`** - the fraction of weights to zero
- **`granularity='weight'`** - individual weights, the model keeps its shape
- **`context='local'`** - each layer on its own
- **`criteria=large_final`** - keep the largest final magnitudes
- **`schedule=one_cycle`** - how the sparsity grows to its target

In [ ]:
sp_cb = SparsifyCallback(sparsity=0.5, granularity='weight', context='local',
                         criteria=large_final, schedule=one_cycle)
learn.fit_one_cycle(5, cbs=sp_cb)

Sparsifying weight until a sparsity of 50.00%
Saving Weights at epoch 0


epoch,train_loss,valid_loss,accuracy,time
0,0.179395,0.468179,0.849120,00:07
1,0.280292,0.729538,0.772666,00:07
2,0.214267,0.196754,0.918809,00:08
3,0.123649,0.170752,0.934371,00:06
4,0.073424,0.160134,0.938430,00:07


Sparsity at the end of epoch 0: 1.96%


Sparsity at the end of epoch 1: 20.07%


Sparsity at the end of epoch 2: 45.86%


Sparsity at the end of epoch 3: 49.74%


Sparsity at the end of epoch 4: 50.00%
Final Sparsity: 50.00%

Sparsity Report:
--------------------------------------------------------------------------------
Layer                          Type            Params     Zeros      Sparsity  
--------------------------------------------------------------------------------
0.0                            Conv2d          9,408      4,702         49.98%
0.4.0.conv1                    Conv2d          36,864     18,430        49.99%
0.4.0.conv2                    Conv2d          36,864     18,430        49.99%
0.4.1.conv1                    Conv2d          36,864     18,430        49.99%
0.4.1.conv2                    Conv2d          36,864     18,430        49.99%
0.5.0.conv1                    Conv2d          73,728     36,862        50.00%
0.5.0.conv2                    Conv2d          147,456    73,726        50.00%
0.5.0.downsample.0             Conv2d          8,192      4,094         49.98%
0.5.1.conv1                    Conv2d         

In [ ]:
record("sparsified", learn.model, report(learn, "sparsified"))

sparsified: 93.84% (1387/1478), Wilson 95% [92.50%, 94.96%]
sparsified: 11.44 M parameters, 149 MMACs, 45.9 MB state_dict, 50.0% zeros in conv weights


Half of the convolution weights are zero and nothing else moved: same parameter count, same MACs,
same file on disk. A dense format stores a zero like any other number.

## 3. Prune

`Pruner` removes the filters themselves. It runs once, outside the training loop, and the fit that
follows retrains what is left.

In [ ]:
pruner = Pruner(learn.model, 0.3, 'local', large_final,
                example_inputs=torch.randn(1, 3, 64, 64).to(default_device()))
pruner.prune_model()

record("pruned", learn.model)

Ignoring output layer: 1.8
Total ignored layers: 1


pruned: 5.59 M parameters, 74 MMACs, 22.4 MB state_dict, 48.8% zeros in conv weights


Pruning drops the share of zeros a little: `large_final` removes the lowest-magnitude filters, which
hold more than their share of zeros. The recovery fit re-applies `SparsifyCallback`, so the model ends
it at 50 % zeros again — its `one_cycle` schedule ramps the target back up from 0, which is why the
intermediate epochs print less: a fresh mask, not the one from step 2.

In [ ]:
sp_cb = SparsifyCallback(sparsity=0.5, granularity='weight', context='local',
                         criteria=large_final, schedule=one_cycle)
learn.fit_one_cycle(3, reset_opt=True, cbs=sp_cb)

Sparsifying weight until a sparsity of 50.00%
Saving Weights at epoch 0


epoch,train_loss,valid_loss,accuracy,time
0,0.272020,1.469996,0.773342,00:05
1,0.188387,0.222212,0.911367,00:04
2,0.106858,0.214357,0.920162,00:06


Sparsity at the end of epoch 0: 10.40%


Sparsity at the end of epoch 1: 48.30%


Sparsity at the end of epoch 2: 50.00%
Final Sparsity: 50.00%

Sparsity Report:
--------------------------------------------------------------------------------
Layer                          Type            Params     Zeros      Sparsity  
--------------------------------------------------------------------------------
0.0                            Conv2d          6,468      3,232         49.97%
0.4.0.conv1                    Conv2d          17,424     8,710         49.99%
0.4.0.conv2                    Conv2d          17,424     8,710         49.99%
0.4.1.conv1                    Conv2d          17,424     8,710         49.99%
0.4.1.conv2                    Conv2d          17,424     8,710         49.99%
0.5.0.conv1                    Conv2d          35,244     17,620        49.99%
0.5.0.conv2                    Conv2d          71,289     35,642        50.00%
0.5.0.downsample.0             Conv2d          3,916      1,956         49.95%
0.5.1.conv1                    Conv2d         

In [ ]:
record("pruned + recovery", learn.model, report(learn, "pruned + recovery"))

pruned + recovery: 92.02% (1360/1478), Wilson 95% [90.52%, 93.29%]
pruned + recovery: 5.59 M parameters, 74 MMACs, 22.4 MB state_dict, 50.0% zeros in conv weights


Removing filters halves the parameter count, the MACs and the `state_dict`, and the model ends the
recovery fit at 50 % zeros again. The accuracy after recovery sits inside the baseline's interval.

## 4. Quantize

`Quantizer(backend='pt2e')` captures the model with `torch.export` and rewrites it to compute in INT8.
Static quantization reads the activation ranges off calibration data, so it needs a loader.

In [ ]:
BATCH = 2   # torch.export captures ONE batch size, and 1478 = 739 x 2

model = learn.model.cpu().eval()
quantizer = Quantizer(backend='pt2e', method='static')
qmodel = quantizer.quantize(model, calibration_dl=dls.train.new(bs=BATCH),
                            max_calibration_samples=64)

print(quantizer.spec.as_dict())
print(quant_spec(qmodel).label, "| exportable:", quant_spec(qmodel).exports)

{'backend': 'pt2e', 'method': 'static', 'weight_bits': 8, 'act_bits': 8, 'qscheme': 'per_channel', 'symmetric': True, 'group_size': None, 'layer_bits': None, 'qdq_placement': 'per_op'}
W8A8 | exportable: True


`W8A8` is the resolved precision — 8-bit weights and activations, per-channel scales, a zero-point of
0 everywhere, as `qdq_stats` shows below — and it travels with the model. That graph still stores float
weights and simulates the quantization; the INT8 artifact is the next step's file.

## 5. Export

`export_qdq` writes the quantize/dequantize pairs into the graph instead of folding them away.
`export_onnx` exports the float model, to have something to compare the file against.

In [ ]:
sample, _ = dls.valid.new(bs=BATCH).one_batch()
sample = sample.cpu()

fp32_path = export_onnx(model, sample, tmp/"pets_fp32.onnx", dynamic_batch=False)
qdq_path = export_qdq(qmodel, sample, tmp/"pets_qdq.onnx")

fp32_mb, qdq_mb = fp32_path.stat().st_size/1e6, qdq_path.stat().st_size/1e6
print(f"{fp32_path.name} {fp32_mb:.1f} MB | {qdq_path.name} {qdq_mb:.1f} MB "
      f"({fp32_mb/qdq_mb:.1f}x smaller on disk)")

[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


pets_fp32.onnx 22.4 MB | pets_qdq.onnx 5.9 MB (3.8x smaller on disk)


In [ ]:
print(qdq_stats(qdq_path).as_dict())
print(sum(1 for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))),
      "weight tensors (convolutions and Linear layers) in the model")

{'n_quantize': 36, 'n_dequantize': 58, 'n_per_channel': 22, 'n_nonzero_zero_point': 0, 'n_unquantized_conv_add': 0}
22 weight tensors (convolutions and Linear layers) in the model


`qdq_stats` counts what the exporter actually wrote:

- `n_quantize` / `n_dequantize` — the pairs a runtime reads to build its INT8 kernels.
- `n_per_channel` — one node per quantized weight tensor, which is the count printed above.
- `n_nonzero_zero_point` — 0, what a symmetric quantizer promised, read off the file.
- `n_unquantized_conv_add` — 0: every `Add` reads its inputs through a pair.

`verify_qdq` reports how often the exported graph and the PyTorch model it came from predict the same
class — worth reading only if the reference predictions vary, so this checks first.

In [ ]:
probe, _ = dls.valid.new(bs=256).one_batch()
probe = probe.cpu()

with torch.no_grad():
    reference = torch.cat([qmodel(c).argmax(-1) for c in probe.split(BATCH)])
assert len(reference.unique()) > 1, "the reference predicts a single class: agreement would be vacuous"
print("reference predictions per class:", torch.bincount(reference, minlength=2).tolist())

agreement = verify_qdq(qmodel, qdq_path, probe, n_batches=len(probe)//BATCH)
k, n, z = round(agreement*len(probe)), len(probe), 1.96
p, d = k/n, 1 + z**2/n
c, h = p + z**2/(2*n), z*sqrt(p*(1-p)/n + z**2/(4*n**2))
print(f"argmax agreement: {agreement:.3f} ({k}/{n}), Wilson 95% [{(c-h)/d:.2%}, {(c+h)/d:.2%}]")

reference predictions per class: [182, 74]


argmax agreement: 1.000 (256/256), Wilson 95% [98.52%, 100.00%]


The reference predicts both classes, so the comparison is not vacuous, and the graph answers the same
class on every probe input. This is agreement with the quantized PyTorch model, not accuracy: it says
the file runs at the batch size it was captured with and computes what its source graph computes.

Agreement is not accuracy: the last cell scores the exported file over the whole validation set
through `ONNXModel` (onnxruntime behind a PyTorch-like call).

In [ ]:
onnx_model = ONNXModel(qdq_path)

k = n = 0
for x, y in dls.valid.new(bs=BATCH):
    k += int((onnx_model(x.cpu()).argmax(-1) == y.cpu()).sum()); n += len(y)
z = 1.96; p, d = k/n, 1 + z**2/n
c, h = p + z**2/(2*n), z*sqrt(p*(1-p)/n + z**2/(4*n**2))
print(f"exported INT8 graph: {k/n:.2%} ({k}/{n}), Wilson 95% [{(c-h)/d:.2%}, {(c+h)/d:.2%}]")

exported INT8 graph: 91.95% (1359/1478), Wilson 95% [90.45%, 93.23%]


In [ ]:
print(f"{'stage':<20}{'params':>10}{'MMACs':>8}{'conv zeros':>12}{'state_dict':>12}{'accuracy':>11}")
for s in stages:
    acc = f"{s['acc']:.2%}" if s['acc'] is not None else "-"
    print(f"{s['stage']:<20}{s['params']/1e6:>9.2f}M{s['macs']/1e6:>8.0f}"
          f"{s['zeros']:>12.1%}{s['mb']:>11.1f}M{acc:>11}")
print("MMACs: multiply-accumulates for one 64x64 image | conv zeros: share of zeros in the "
      "convolution weights")
print(f"\nONNX files: FP32 {fp32_mb:.1f} MB | QDQ INT8 {qdq_mb:.1f} MB, "
      f"which scored {k/n:.2%} ({k}/{n}) on the validation set")

stage                   params   MMACs  conv zeros  state_dict   accuracy
baseline                11.44M     149        0.0%       45.9M     92.22%
sparsified              11.44M     149       50.0%       45.9M     93.84%
pruned                   5.59M      74       48.8%       22.4M          -
pruned + recovery        5.59M      74       50.0%       22.4M     92.02%
MMACs: multiply-accumulates for one 64x64 image | conv zeros: share of zeros in the convolution weights

ONNX files: FP32 22.4 MB | QDQ INT8 5.9 MB, which scored 91.95% (1359/1478) on the validation set


The four accuracies of this run are successive stages of one model, not the arms of an A/B — each row
carries a different training budget — and their Wilson intervals overlap, so this page ranks none of
them.

**Scope.** ResNet-18 with ImageNet weights, PETS cat/dog labels, 64 px: three epochs of fine-tuning,
five with `SparsifyCallback`, one `Pruner` step, three recovery epochs, calibration on 64 training
images. Single run, no seed fixed; device and library versions printed at the top, the ONNX graphs
scored on CPU through onnxruntime. Accuracy is measured on the 1478 validation images with its Wilson
95% interval. This page measures no latency.

## Summary

| Step | Tool | What it gives you |
|------|------|-------------------|
| Sparsify | `SparsifyCallback(sparsity, granularity, context, criteria, schedule)` | Weights zeroed during training, following the schedule |
| Prune | `Pruner(model, pruning_ratio, context, criteria)` | Filters removed, a structurally smaller model |
| Quantize | `Quantizer(backend='pt2e', method='static').quantize(model, calibration_dl)` | An INT8 graph, tagged with the precision that produced it |
| Export | `export_qdq(model, sample, path)` | An ONNX file whose Q/DQ pairs survived the export |
| Inspect | `qdq_stats(path)` | Node counts, per-channel count, zero-point audit |
| Verify | `verify_qdq(model, path, samples)` | Argmax agreement between the exported graph and PyTorch |
| Score | `ONNXModel(path)` | The exported file called like a PyTorch model |
| Cost | `tp.utils.count_ops_and_params` | MACs and parameters, at a given input size |

`sparsity` and `pruning_ratio` are fractions in [0, 1]. `criteria` and `schedule` take the objects
exported by `fasterai.core.criteria` and `fasterai.core.schedule`.

---

## See Also

- [Sparsify Callback](sparse/sparsify_callback.html) - The sparsification step on its own
- [Prune Callback](prune/prune_callback.html) - Pruning inside the training loop instead of between fits
- [Deployable INT8 Export](quantize/deployable_export.html) - The precision grammar and the QDQ export in detail
- [Knowledge Distillation](distill/distill_callback.html) - Training a student model from a teacher
- [BatchNorm Folding](misc/bn_folding.html) - Folding BatchNorm layers into their convolutions
- [FC Decomposition](misc/fc_decomposer.html) - Factorizing fully-connected layers
- [Sensitivity Analysis](analyze/sensitivity.html) - Per-layer compression targets